In [ ]:
# Run once in a fresh notebook environment.
%pip install -q numpy scipy pandas matplotlib


# Round 4 Reproduction

**Purpose.** Reproduce the deliberately constructed environments in which a manual active-search true-equal-outcome policy should outperform immediate equal split, then test whether the RR approximation discovers comparable behavior.

The reported grid contains 972 environments and compares RR, manual active search, and equal split on common episodes.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the repository checkout.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print(PROJECT_ROOT)


In [ ]:
RUN = False
EPISODES = 1200
VOI_SAMPLES = 500
OBSERVATIONS_PER_PERSON = 500
MANUAL_SAMPLES_PER_PERSON = 3
GRID_CHUNKS = 81
MAX_WORKERS = max(1, (os.cpu_count() or 2) - 1)
OUTPUT_DIR = "results/round_04_notebook/active_search_benchmark"
print({"run": RUN, "episodes": EPISODES, "voi_samples": VOI_SAMPLES, "manual_samples_per_person": MANUAL_SAMPLES_PER_PERSON})


## Constructed active-search benchmark

**Test purpose.** Establish that manual active-search equal outcome beats equal split before evaluating whether the RR approximation reaches similar true-state equal-outcome behavior and expected utility.


In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "run_parallel_experiments.py"),
    "--preset", "server",
    "--sections", "active_search_diagnostic",
    "--regime-grid", "active_search_benchmark",
    "--regime-grid-chunks", str(GRID_CHUNKS),
    "--episodes", str(EPISODES),
    "--voi-samples", str(VOI_SAMPLES),
    "--common-observations", "on",
    "--observations-per-person", str(OBSERVATIONS_PER_PERSON),
    "--manual-active-samples-per-person", str(MANUAL_SAMPLES_PER_PERSON),
    "--max-workers", str(MAX_WORKERS),
    "--output-dir", OUTPUT_DIR,
]
print(" ".join(command))
if RUN:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)


## Inspect policy and environment summaries

Expected average utility is the performance criterion. True equal-outcome rates, distance to the true equal-outcome allocation, distance from 50/50, and sample count diagnose behavior.


In [ ]:
import pandas as pd

output_dir = PROJECT_ROOT / OUTPUT_DIR
for filename in ("active_search_diagnostic_environment_summary.csv", "active_search_diagnostic_manual_advantage_candidates.csv", "active_search_diagnostic_policy_profiles.csv"):
    matches = list(output_dir.rglob(filename)) if output_dir.exists() else []
    print(f"{filename}: {len(matches)} file(s)")
    if matches:
        display(pd.read_csv(matches[0]).head(20))
